# ClaraVis — Fine-Tuning do YOLO para Navegacao Assistiva

Este notebook treina o modelo YOLO para detectar objetos importantes para
pessoas com deficiencia visual: escadas, buracos, faixas de pedestre, obstaculos, etc.

**ANTES DE COMECAR:**
1. Clique em **Runtime > Change runtime type > T4 GPU > Save**
2. Tenha sua API Key do Roboflow (veja o GUIA_FINE_TUNING.md)

Rode cada celula na ordem clicando no botao ▶ (play).

## Celula 1 — Instalar dependencias
Rode esta celula primeiro. Vai instalar tudo que precisamos.

In [ ]:
!pip install ultralytics roboflow -q
import torch
print(f"GPU disponivel: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("ATENCAO: GPU nao detectada! Va em Runtime > Change runtime type > T4 GPU")

## Celula 2 — Configurar sua API Key do Roboflow

Substitua `SUA_API_KEY_AQUI` pela sua chave do Roboflow.
Para pegar: Roboflow.com > Settings > Roboflow API > copie a Private API Key.

In [ ]:
# ========================================
# COLE SUA API KEY AQUI (entre as aspas)
# ========================================
ROBOFLOW_API_KEY = "SUA_API_KEY_AQUI"

# Resolucao de treino (416 = melhor deteccao, 320 = mais rapido)
IMG_SIZE = 416

# Epocas de treino (100 = bom ponto de partida)
EPOCHS = 100

print(f"Configurado! Resolucao: {IMG_SIZE}x{IMG_SIZE}, Epocas: {EPOCHS}")

## Celula 3 — Baixar o dataset

Esta celula baixa imagens de obstaculos em calcadas do Roboflow.
Se quiser usar outro dataset, substitua workspace/project/version abaixo.

**Para encontrar datasets:**
1. Va em https://universe.roboflow.com
2. Pesquise: `sidewalk obstacle`, `pothole`, `crosswalk`, `stairs`
3. Clique no dataset > Download > YOLOv8 > Show download code
4. Copie workspace, project e version do codigo gerado

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# ========================================
# SUBSTITUA ABAIXO COM SEU DATASET
# ========================================
# Exemplo: dataset de obstaculos em calcadas
# Voce pega esses valores no "download code" do Roboflow Universe
#
# Se nao sabe qual usar, va em universe.roboflow.com e busque:
#   "sidewalk obstacle detection" ou "pothole detection"
# Clique no dataset > Download > YOLOv8 > Show download code

WORKSPACE = "seu-workspace"     # Ex: "daniel-claravis"
PROJECT = "seu-project"         # Ex: "sidewalk-obstacles"
VERSION = 1                     # Ex: 1, 2, 3...

try:
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download("yolov8")
    DATA_YAML = dataset.location + "/data.yaml"
    print(f"\nDataset baixado com sucesso!")
    print(f"Local: {dataset.location}")
    print(f"data.yaml: {DATA_YAML}")

    # Mostrar info do dataset
    with open(DATA_YAML) as f:
        print(f"\nConteudo do data.yaml:")
        print(f.read())
except Exception as e:
    print(f"Erro: {e}")
    print()
    print("=" * 50)
    print("INSTRUCOES:")
    print("1. Va em universe.roboflow.com")
    print("2. Busque um dataset (ex: 'sidewalk obstacle')")
    print("3. Clique nele > Download > YOLOv8 > Show download code")
    print("4. Copie os valores de workspace, project, version")
    print("5. Cole acima e rode esta celula novamente")
    print("=" * 50)
    DATA_YAML = None

## Celula 4 — TREINAR O MODELO (principal!)

Esta e a celula mais importante. Ela treina o YOLO com as suas imagens.

**Vai demorar 20-60 minutos.** NAO feche o navegador!

Voce vai ver o progresso: `Epoch 1/100`, `Epoch 2/100`, etc.
Observe o `mAP50` — quanto maior, melhor (acima de 0.5 ja e util).

In [ ]:
from ultralytics import YOLO

if DATA_YAML is None:
    print("ERRO: Voce precisa baixar o dataset primeiro (Celula 3)")
    print("Configure WORKSPACE, PROJECT e VERSION e rode a Celula 3 novamente")
else:
    # Carregar modelo base (pre-treinado no COCO com 80 classes)
    model = YOLO("yolov8s.pt")

    # TREINAR!
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=16,             # Se der erro de memoria, mude para 8
        device=0,             # GPU
        project="claravis",
        name="treino",

        # Augmentacoes para melhor resultado em ambientes reais
        augment=True,
        mosaic=1.0,           # Combina 4 imagens (ajuda muito)
        hsv_h=0.02,           # Variacao de cor
        hsv_s=0.7,            # Variacao de saturacao
        hsv_v=0.5,            # Variacao de brilho (IMPORTANTE para pouca luz)
        flipud=0.0,           # NAO inverter vertical (gravidade importa)
        fliplr=0.5,           # Inverter horizontal OK
        degrees=5.0,          # Rotacao leve (camera na cabeca oscila)
        scale=0.5,            # Escala (objetos em distancias variadas)

        # Otimizacoes
        lr0=0.01,
        warmup_epochs=3,
        weight_decay=0.0005,
    )

    print("\n" + "=" * 50)
    print("TREINO FINALIZADO!")
    print("=" * 50)

## Celula 5 — Ver os resultados

Mostra graficos de como o modelo aprendeu e exemplos de deteccao.

In [ ]:
from IPython.display import Image, display
import glob

# Mostrar graficos de treino
results_dir = "claravis/treino"
print("=== GRAFICOS DE APRENDIZADO ===\n")

for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    img_path = f"{results_dir}/{img_name}"
    try:
        display(Image(filename=img_path, width=800))
        print(f"\n{img_name}\n")
    except:
        print(f"Imagem nao encontrada: {img_name}")

# Mostrar metricas finais
print("\n=== COMO INTERPRETAR ===")
print("- mAP50: precisao media a 50% IoU (acima de 0.5 = bom, acima de 0.7 = otimo)")
print("- Precision: de tudo que detectou, quanto estava certo")
print("- Recall: de tudo que existia, quanto encontrou")
print("- Se Precision alta mas Recall baixo: modelo conservador (perde objetos)")
print("- Se Recall alto mas Precision baixa: modelo liberal (detecta coisas erradas)")

## Celula 6 — Exportar para o celular (TFLite)

Converte o modelo treinado para o formato que o celular entende.

In [ ]:
from pathlib import Path
import shutil

best_model_path = "claravis/treino/weights/best.pt"

if not Path(best_model_path).exists():
    print("ERRO: Modelo nao encontrado. Voce rodou a Celula 4 (treino)?")
else:
    trained = YOLO(best_model_path)

    # Exportar 416x416 float16 (principal)
    print("Exportando modelo 416x416 float16...")
    trained.export(format="tflite", imgsz=416, half=True)

    # Copiar com nome amigavel
    export_dir = Path("claravis/treino/weights/best_saved_model")
    tflite_files = list(export_dir.rglob("*.tflite")) if export_dir.exists() else []

    if tflite_files:
        src = tflite_files[0]
        dst = Path("claravis_model_416.tflite")
        shutil.copy(src, dst)
        size_mb = dst.stat().st_size / 1024 / 1024
        print(f"\nModelo exportado: {dst} ({size_mb:.1f} MB)")
    else:
        print("Procurando modelo exportado...")
        for f in Path(".").rglob("*float16.tflite"):
            print(f"  Encontrado: {f} ({f.stat().st_size/1024/1024:.1f} MB)")

    print("\nPronto! Agora rode a Celula 7 para baixar o arquivo.")

## Celula 7 — Baixar o modelo para o seu computador

Clique no link que aparecer para baixar o arquivo `.tflite`.
Depois, siga as instrucoes do guia para copiar para o celular.

In [ ]:
from google.colab import files
from pathlib import Path

# Baixar o modelo treinado
model_file = Path("claravis_model_416.tflite")
if model_file.exists():
    print(f"Baixando {model_file.name} ({model_file.stat().st_size/1024/1024:.1f} MB)...")
    files.download(str(model_file))
    print("\nDepois de baixar, copie para o celular:")
    print("  Opcao A (com cabo): adb push claravis_model_416.tflite /sdcard/Download/claravis_model.tflite")
    print("  Opcao B (sem cabo): envie o arquivo para o celular e coloque na pasta Downloads")
else:
    print("Modelo nao encontrado. Rode as celulas anteriores primeiro.")
    # Tentar encontrar qualquer tflite
    for f in Path(".").rglob("*.tflite"):
        print(f"  Encontrado: {f}")

# Tambem baixar o modelo .pt (para futuras iteracoes)
best_pt = Path("claravis/treino/weights/best.pt")
if best_pt.exists():
    print(f"\nBaixando tambem o best.pt ({best_pt.stat().st_size/1024/1024:.1f} MB)...")
    print("(Guarde este arquivo — e o modelo treinado completo, util para retreino)")
    files.download(str(best_pt))